In [1]:
pip install fpdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import random
from faker import Faker
from datetime import datetime, timedelta
import pandas as pd


c:\anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
import os
import json
import random
import pandas as pd
from faker import Faker

fake = Faker("fr_FR")

random.seed(42)
Faker.seed(42)

DATASET_DIR = "../dataset"
OUTPUT_FILE = os.path.join(DATASET_DIR, "documents_dataset.json")

os.makedirs(DATASET_DIR, exist_ok=True)

print("Dataset sera sauvegardé ici :", OUTPUT_FILE)

def generate_siret(valid=True):
    if valid:
        return fake.siret()
    else:
        choices = [
            None,
            str(random.randint(1000000000000, 9999999999999)),  # 13 chiffres
            str(random.randint(100000000000000, 999999999999999)),  # 15 chiffres
            "12345678901234"  # checksum faux
        ]
        return random.choice(choices)

def generate_tva(siret, valid=True):
    if not valid or not siret:
        return "FR00000000000"
    return "FR" + siret[:11]

def generate_iban(valid=True):
    if valid:
        return fake.iban()
    else:
        return random.choice([
            "FR123",  # trop court
            "INVALIDIBAN123456",
            None
        ])

def generate_bic(valid=True):
    if valid:
        return fake.swift()
    else:
        return random.choice([
            "ABC",  # trop court
            "INVALIDBICCODE123",
            None
        ])

def future_date():
    return fake.date_between(start_date="+30d", end_date="+365d")

def past_date():
    return fake.date_between(start_date="-365d", end_date="-30d")

def generate_facture(valid=True):
    siret = generate_siret(valid)
    montant_ht = round(random.uniform(100, 5000), 2)

    facture = {
        "document_type": "facture",
        "company": fake.company(),
        "siret": siret,
        "tva": generate_tva(siret, valid),
        "date_emission": str(fake.date()),
        "montant_ht": montant_ht,
        "montant_ttc": round(montant_ht * 1.2, 2),
        "status": "valid" if valid else "anomaly"
    }

    if not valid:
        anomaly_type = random.choice([
            "missing_siret",
            "wrong_tva",
            "missing_amount"
        ])

        if anomaly_type == "missing_siret":
            facture["siret"] = None

        elif anomaly_type == "wrong_tva":
            facture["tva"] = "FR99999999999"

        elif anomaly_type == "missing_amount":
            facture["montant_ht"] = None

    return facture

def generate_devis(valid=True):
    siret = generate_siret(valid)
    montant_ht = round(random.uniform(100, 3000), 2)

    devis = {
        "document_type": "devis",
        "company": fake.company(),
        "siret": siret,
        "date": str(fake.date()),
        "montant_ht": montant_ht,
        "montant_ttc": round(montant_ht * 1.2, 2)
    }

    if not valid:
        devis["siret"] = None

    return devis

def generate_attestation_siret(valid=True):
    return {
        "document_type": "attestation_siret",
        "company": fake.company(),
        "siret": generate_siret(valid),
        "date_emission": str(fake.date())
    }

def generate_urssaf(valid=True):
    siret = generate_siret(valid)
    expiration = future_date() if valid else past_date()

    return {
        "document_type": "attestation_urssaf",
        "company": fake.company(),
        "siret": siret,
        "date_emission": str(fake.date()),
        "date_expiration": str(expiration),
        "status": "valid" if valid else "expired"
    }

def generate_kbis(valid=True):
    return {
        "document_type": "kbis",
        "company": fake.company(),
        "siret": generate_siret(valid),
        "date_creation": str(fake.date_between(start_date="-10y", end_date="-1y")),
        "adresse": fake.address() if valid else None,
        "dirigeant": fake.name()
    }

def generate_rib(valid=True):
    return {
        "document_type": "rib",
        "company": fake.company(),
        "iban": generate_iban(valid),
        "bic": generate_bic(valid),
        "bank": fake.company() + " Bank"
    }

dataset = []

for _ in range(200):
    dataset.append(generate_facture(valid=True))

for _ in range(100):
    dataset.append(generate_facture(valid=False))

for _ in range(150):
    dataset.append(generate_devis(valid=True))

for _ in range(50):
    dataset.append(generate_devis(valid=False))

for _ in range(100):
    dataset.append(generate_attestation_siret(valid=True))

for _ in range(50):
    dataset.append(generate_attestation_siret(valid=False))

for _ in range(80):
    dataset.append(generate_urssaf(valid=True))

for _ in range(40):
    dataset.append(generate_urssaf(valid=False))

for _ in range(100):
    dataset.append(generate_kbis(valid=True))

for _ in range(50):
    dataset.append(generate_kbis(valid=False))

for _ in range(100):
    dataset.append(generate_rib(valid=True))

for _ in range(50):
    dataset.append(generate_rib(valid=False))

print("Nombre total de documents :", len(dataset))

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=4, ensure_ascii=False)

print("Dataset sauvegardé")

df = pd.DataFrame(dataset)
print(df.head())

Dataset sera sauvegardé ici : ../dataset\documents_dataset.json
Nombre total de documents : 1070
Dataset sauvegardé
  document_type         company              siret            tva  \
0       facture  Maury S.A.R.L.  104 332 184 00811  FR104 332 184   
1       facture         Vasseur  133 890 830 00036  FR133 890 830   
2       facture           Henry  402 654 230 00752  FR402 654 230   
3       facture      Hoareau SA  155 940 786 00488  FR155 940 786   
4       facture          Rey SA  959 310 343 00378  FR959 310 343   

  date_emission  montant_ht  montant_ttc status date date_expiration  \
0    1972-02-29     3233.19      3879.83  valid  NaN             NaN   
1    2000-07-27      222.55       267.06  valid  NaN             NaN   
2    1976-04-24     1447.64      1737.17  valid  NaN             NaN   
3    2007-07-27     1193.73      1432.48  valid  NaN             NaN   
4    1995-11-14     3708.71      4450.45  valid  NaN             NaN   

  date_creation adresse dirigeant ib

In [4]:
# 📄 Génération des PDFs
from fpdf import FPDF
import hashlib

# Création des dossiers pour les PDFs
PDF_DIR = os.path.join(DATASET_DIR, "pdfs")
for doc_type in ["facture", "devis", "attestation_siret", "attestation_urssaf", "kbis", "rib"]:
    os.makedirs(os.path.join(PDF_DIR, doc_type), exist_ok=True)

def create_pdf(document):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    # Titre et entreprise
    pdf.cell(0, 10, f"Type de document: {document.get('document_type','')}", ln=True)
    pdf.cell(0, 10, f"Entreprise: {document.get('company','')}", ln=True)

    # Champs spécifiques par type
    dtype = document.get("document_type","")
    
    if dtype in ["facture", "devis"]:
        pdf.cell(0, 10, f"SIRET: {document.get('siret','')}", ln=True)
        pdf.cell(0, 10, f"TVA: {document.get('tva','')}", ln=True)
        pdf.cell(0, 10, f"Date: {document.get('date_emission', document.get('date',''))}", ln=True)
        pdf.cell(0, 10, f"Montant HT: {document.get('montant_ht','')}", ln=True)
        pdf.cell(0, 10, f"Montant TTC: {document.get('montant_ttc','')}", ln=True)
        if "status" in document:
            pdf.cell(0, 10, f"Status: {document.get('status','')}", ln=True)
    elif dtype == "attestation_siret":
        pdf.cell(0, 10, f"SIRET: {document.get('siret','')}", ln=True)
        pdf.cell(0, 10, f"Date émission: {document.get('date_emission','')}", ln=True)
    elif dtype == "attestation_urssaf":
        pdf.cell(0, 10, f"SIRET: {document.get('siret','')}", ln=True)
        pdf.cell(0, 10, f"Date émission: {document.get('date_emission','')}", ln=True)
        pdf.cell(0, 10, f"Date expiration: {document.get('date_expiration','')}", ln=True)
        pdf.cell(0, 10, f"Status: {document.get('status','')}", ln=True)
    elif dtype == "kbis":
        pdf.cell(0, 10, f"SIRET: {document.get('siret','')}", ln=True)
        pdf.cell(0, 10, f"Date création: {document.get('date_creation','')}", ln=True)
        pdf.cell(0, 10, f"Adresse: {document.get('adresse','')}", ln=True)
        pdf.cell(0, 10, f"Dirigeant: {document.get('dirigeant','')}", ln=True)
    elif dtype == "rib":
        pdf.cell(0, 10, f"IBAN: {document.get('iban','')}", ln=True)
        pdf.cell(0, 10, f"BIC: {document.get('bic','')}", ln=True)
        pdf.cell(0, 10, f"Banque: {document.get('bank','')}", ln=True)

    # Nom du fichier unique
    hash_name = hashlib.md5(str(document).encode()).hexdigest()
    file_path = os.path.join(PDF_DIR, dtype, f"{hash_name}.pdf")
    pdf.output(file_path)

# Création de tous les PDFs
for doc in dataset:
    create_pdf(doc)

print("Tous les PDFs ont été générés dans le dossier :", PDF_DIR)

Tous les PDFs ont été générés dans le dossier : ../dataset\pdfs


In [5]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\tesseract\tesseract.exe"

print(pytesseract.get_tesseract_version())

5.5.0.20241111


In [6]:
pip install pytesseract

Note: you may need to restart the kernel to use updated packages.
